In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import train_test_split,StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder
import joblib

#import classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier,RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')


# Classification

In [225]:
df=pd.read_csv("healthcare-dataset-stroke-data.csv")
df.shape

(5110, 12)

In [226]:
df

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,Female,80.0,1,0,Yes,Private,Urban,83.75,NaN,never smoked,0
5106,44873,Female,81.0,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35.0,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51.0,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0


In [227]:
df=df.drop_duplicates()
df.shape

(5110, 12)

In [228]:
df.isna().sum()

id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

In [229]:
print(df.shape)
df=df.dropna(subset=['bmi'])
print(df.shape)

(5110, 12)
(4909, 12)


### Define the Pipeline Classes

In [169]:
class OutlierHandler:
    def __init__(self, columns):
        self.columns = columns
        self.bounds = {}

    def fit(self, df):
        for col in self.columns:
            if col not in df.columns:
                continue

            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1

            self.bounds[col] = {
                "LOF": Q1 - 1.5* IQR,
                "UOF": Q3 + 1.5 * IQR
            }
        return self

    def transform(self, df):
        df = df.copy()

        for col in self.columns:
            if col not in df.columns or col not in self.bounds:
                continue

            LOF = self.bounds[col]["LOF"]
            UOF = self.bounds[col]["UOF"]

            df[col + "_flag"] = df[col].apply(
                lambda x: 1 if x < LOF or x > UOF else 0
            )
            df[col] = df[col].clip(lower=LOF, upper=UOF)

        return df


class FullDataPipeline:

    def __init__(self, outlier_cols,
                 apply_scaling=False,
                 handle_outliers=True,
                 apply_encoding=True,
                 drop_missing=True):

        self.apply_scaling = apply_scaling
        self.handle_outliers = handle_outliers
        self.apply_encoding = apply_encoding
        self.drop_missing = drop_missing

        self.outlier_handler = OutlierHandler(outlier_cols) if handle_outliers else None

        try:
            self.ohe = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
        except TypeError:
            self.ohe = OneHotEncoder(drop="first", handle_unknown="ignore", sparse=False)

        self.numeric_scaler = None
        self.train_columns = None

        self.categorical_cols = [
            "gender",
            "ever_married",
            "work_type",
            "Residence_type",
            "smoking_status"
        ]

        self.required_cols = ["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi"]

    def _validate_columns(self, df, stage="transform"):
        missing = set(self.required_cols) - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns at {stage}: {missing}")

    def _clean_basic(self, df):
        df = df.copy()
        df.dropna(how="all", inplace=True)
        df.drop(columns=["id"], errors="ignore", inplace=True)

        if self.drop_missing:
            df.dropna(axis=0, how="any", inplace=True)

        return df

    def _scale_numeric(self, df, fit=False):
        df = df.copy()

        if not self.apply_scaling:
            return df

        num_cols = df.select_dtypes(include=["number"]).columns.tolist()
        num_cols=[col for col in num_cols if col not in self.categorical_cols]

        if fit:
            self.numeric_scaler = StandardScaler()
            df[num_cols] = self.numeric_scaler.fit_transform(df[num_cols])
        else:
            df[num_cols] = self.numeric_scaler.transform(df[num_cols])

        return df

    def _encode(self, df, fit=False):
        cat_cols = [c for c in self.categorical_cols if c in df.columns]
        num_cols = [c for c in df.columns if c not in cat_cols]

        df_num = df[num_cols].copy()

        if not self.apply_encoding:
            if len(cat_cols) == 0:
                return df_num
            return pd.concat([df_num, df[cat_cols].copy()], axis=1)

        if len(cat_cols) == 0:
            return df_num

        if fit:
            ohe_array = self.ohe.fit_transform(df[cat_cols])
        else:
            ohe_array = self.ohe.transform(df[cat_cols])

        ohe_names = self.ohe.get_feature_names_out(cat_cols)
        df_cat = pd.DataFrame(ohe_array, columns=ohe_names, index=df.index)

        return pd.concat([df_num, df_cat], axis=1)

    def fit(self, df):
        df = self._clean_basic(df)

        if "stroke" not in df.columns:
            raise ValueError("Target column 'stroke' not found in training data")
        y = df.pop("stroke")

        self._validate_columns(df, stage="fit")

        if self.handle_outliers:
            self.outlier_handler.fit(df)
            df = self.outlier_handler.transform(df)

        df = self._scale_numeric(df, fit=True)

        df = self._encode(df, fit=True)

        self.train_columns = df.columns.tolist()
        return df, y

    def transform(self, df):
        if self.train_columns is None:
            raise ValueError("Pipeline has not been fitted yet. Call fit() first.")

        df = self._clean_basic(df)

        y = None
        if "stroke" in df.columns:
            y = df.pop("stroke")

        self._validate_columns(df, stage="transform")

        if self.handle_outliers:
            df = self.outlier_handler.transform(df)

        df = self._scale_numeric(df, fit=False)

        df = self._encode(df, fit=False)

        df = df.reindex(columns=self.train_columns, fill_value=0)

        return df, y

    def fit_transform(self, df):
        return self.fit(df)


In [170]:
# ═══════════════════════════════════════════════════════════════
#  PIPELINES
# ═══════════════════════════════════════════════════════════════
if __name__ == "__main__":

    # Based on EDA: outliers mainly in BMI
    outlier_columns = ["bmi"]

    # ─────────────────────────────────────────────────────────────
    #  1:  CatBoost
    # ─────────────────────────────────────────────────────────────
    print("1: Pipeline for CatBoost")
    print("=" * 60)

    pipeline_catboost = FullDataPipeline(
        outlier_cols=outlier_columns,
        apply_scaling=False,
        handle_outliers=False,  
        apply_encoding=False,    
        drop_missing=False
    )

    print("Configuration:")
    print("  - Scaling: DISABLED")
    print("  - Outlier handling: DISABLED")
    print("  - Encoding: DISABLED (CatBoost handles categoricals natively)")
    print("  - Missing value dropped: DISABLED")
    print()

    # ─────────────────────────────────────────────────────────────
    # 2:  XGBoost / LightGBM
    # ─────────────────────────────────────────────────────────────
    print("2: Pipeline for XGBoost / LightGBM")
    print("=" * 60)

    pipeline_xgb = FullDataPipeline(
        outlier_cols=outlier_columns,
        apply_scaling=False,
        handle_outliers=False,  
        apply_encoding=True,     
        drop_missing=False
    )

    print("Configuration:")
    print("  - Scaling: DISABLED")
    print("  - Outlier handling: DISABLED")
    print("  - Encoding: ENABLED")
    print("  - Missing value dropped: DISABLED")
    print()

    # ─────────────────────────────────────────────────────────────
    # 3: Linear Models 
    # ─────────────────────────────────────────────────────────────
    print("3: Pipeline for Linear Models")
    print("=" * 60)

    pipeline_linear = FullDataPipeline(
        outlier_cols=outlier_columns,
        apply_scaling=True,
        handle_outliers=True,    
        apply_encoding=True,
        drop_missing=True
    )

    print("Configuration:")
    print("  - Scaling: ENABLED")
    print("  - Outlier handling: ENABLED (BMI only)")
    print("  - Encoding: ENABLED")
    print("  - Missing values: DROPPED")
    print("  - All preprocessing ENABLED")
    print()

    # ─────────────────────────────────────────────────────────────
    # TREE BASED(ROBUST)
    # ─────────────────────────────────────────────────────────────
    print("4: Pipeline for TREE BASED(ROBUST)")
    print("=" * 60)

    pipeline_tree_robust = FullDataPipeline(
        outlier_cols=outlier_columns,
        apply_scaling=False,
        handle_outliers=False,   
        apply_encoding=True,
        drop_missing=True
    )

    print("Configuration:")
    print("  - Scaling: DISABLED")
    print("  - Outlier handling: DISABLED")
    print("  - Encoding: ENABLED")
    print("  - Missing values: DROPPED")
    print()

    print("5: Pipeline for  ADABOOST")
    print("=" * 60)
    pipeline_tree_sensitive = FullDataPipeline(
        outlier_cols=outlier_columns,
        apply_scaling=False,    
        drop_missing=True,    
        handle_outliers=True,
        apply_encoding=True)
        
    print("Configuration:")
    print("  - Scaling: DISABLED")
    print("  - Missing value dropped: ENABLED")
    print("  - Outlier handling: ENABLED")
    print("  - Encoding: ENABLED")
    print()


1: Pipeline for CatBoost
Configuration:
  - Scaling: DISABLED
  - Outlier handling: DISABLED
  - Encoding: DISABLED (CatBoost handles categoricals natively)
  - Missing value dropped: DISABLED

2: Pipeline for XGBoost / LightGBM
Configuration:
  - Scaling: DISABLED
  - Outlier handling: DISABLED
  - Encoding: ENABLED
  - Missing value dropped: DISABLED

3: Pipeline for Linear Models
Configuration:
  - Scaling: ENABLED
  - Outlier handling: ENABLED (BMI only)
  - Encoding: ENABLED
  - Missing values: DROPPED
  - All preprocessing ENABLED

4: Pipeline for TREE BASED(ROBUST)
Configuration:
  - Scaling: DISABLED
  - Outlier handling: DISABLED
  - Encoding: ENABLED
  - Missing values: DROPPED

5: Pipeline for  ADABOOST
Configuration:
  - Scaling: DISABLED
  - Missing value dropped: ENABLED
  - Outlier handling: ENABLED
  - Encoding: ENABLED



In [171]:
target = 'stroke'
#First split → Train + Temp
df_train, df_test = train_test_split(
    df,
    test_size=0.20,                
    stratify=df[target],            
    random_state=42)

#check distribustion
print("Train distribution:\n", df_train[target].value_counts(normalize=True))
print("\nTest distribution:\n", df_test[target].value_counts(normalize=True))


Train distribution:
 stroke
0    0.957474
1    0.042526
Name: proportion, dtype: float64

Test distribution:
 stroke
0    0.95723
1    0.04277
Name: proportion, dtype: float64


#### 1-Gradient Boosting  (Xgboost,LightGBM)

In [172]:
pipeline_gradient_boosting  = FullDataPipeline(
        outlier_cols=outlier_columns,
        apply_scaling=False,
        handle_outliers=False,   
        drop_missing=False )

In [173]:
X_train_gb, y_train_gb = pipeline_gradient_boosting.fit(df_train)
X_test_gb, y_test_gb = pipeline_gradient_boosting.transform(df_test)

In [174]:
X_train_gb.columns

Index(['age', 'avg_glucose_level', 'bmi', 'gender_Male', 'gender_Other',
       'ever_married_Yes', 'work_type_Never_worked', 'work_type_Private',
       'work_type_Self-employed', 'work_type_children', 'Residence_type_Urban',
       'smoking_status_formerly smoked', 'smoking_status_never smoked',
       'smoking_status_smokes', 'hypertension_1', 'heart_disease_1'],
      dtype='object')

In [175]:
X_train_gb.to_csv("X_train_gb.csv", index=False)
y_train_gb.to_csv("y_train_gb.csv", index=False)
X_test_gb.to_csv("X_test_gb.csv", index=False)
y_test_gb.to_csv("y_test_gb.csv", index=False)

#### 2- Linear Models

In [176]:
pipeline_linear = FullDataPipeline(
        outlier_cols=outlier_columns,
        apply_scaling=True,
        handle_outliers=True,    
        apply_encoding=True,
        drop_missing=True
    )

In [177]:
X_train_s, y_train = pipeline_linear.fit(df_train)
X_test_s, y_test = pipeline_linear.transform(df_test)

In [178]:
X_train_s

,age,avg_glucose_level,bmi,bmi_flag,gender_Male,gender_Other,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,hypertension_1,heart_disease_1
1473,0.266654,0.750317,1.457090,-0.035705,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
4539,0.891646,2.188165,-0.326006,-0.035705,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
953,-1.652963,-0.286120,-1.295079,-0.035705,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2563,0.623792,-0.343688,-0.313085,-0.035705,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2426,-1.608320,-0.585114,-1.734392,-0.035705,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1939,-0.447622,-1.068413,-0.855766,-0.035705,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
757,-1.072613,-0.479574,0.371727,-0.035705,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
5024,-0.447622,1.060690,0.332965,-0.035705,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1853,0.534508,2.065220,0.022861,-0.035705,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


In [179]:
X_train_s.describe().T

,count,mean,std,min,25%,50%,75%,max
age,3927.0,9.680172e-17,1.000127,-1.917245,-0.804760,0.043443,0.802362,1.739849
avg_glucose_level,3927.0,2.207441e-16,1.000127,-1.130889,-0.639558,-0.311334,0.195728,3.702546
bmi,3927.0,4.016819e-16,1.000127,-2.406284,-0.687793,-0.106349,0.539700,4.222180
bmi_flag,3927.0,6.332823e-18,1.000127,-0.035705,-0.035705,-0.035705,-0.035705,28.007142
gender_Male,3927.0,4.036160e-01,0.490685,0.000000,0.000000,0.000000,1.000000,1.000000
gender_Other,3927.0,2.546473e-04,0.015958,0.000000,0.000000,0.000000,0.000000,1.000000
ever_married_Yes,3927.0,6.574994e-01,0.474606,0.000000,0.000000,1.000000,1.000000,1.000000
work_type_Never_worked,3927.0,4.329004e-03,0.065661,0.000000,0.000000,0.000000,0.000000,1.000000
work_type_Private,3927.0,5.793226e-01,0.493731,0.000000,0.000000,1.000000,1.000000,1.000000
work_type_Self-employed,3927.0,1.591546e-01,0.365867,0.000000,0.000000,0.000000,0.000000,1.000000


In [180]:
X_train_s.to_csv("X_train_s.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
X_test_s.to_csv("X_test_s.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

#### 3-Tree Base (ROBUST)  Model

In [181]:
X_train_tree, y_train_tree = pipeline_tree_robust.fit(df_train)
X_test_tree, y_test_tree = pipeline_tree_robust.transform(df_test)

In [182]:
X_train_tree.to_csv("X_train_tree.csv", index=False)
y_train_tree.to_csv("y_train_tree.csv", index=False)
X_test_tree.to_csv("X_test_tree.csv", index=False)
y_test_tree.to_csv("y_test_tree.csv", index=False)

#### 4-Catboost Model

In [183]:
X_train_ca, y_train_ca= pipeline_catboost.fit(df_train)
X_test_ca, y_test_ca= pipeline_catboost.transform(df_test)

In [184]:
X_train_ca.to_csv("X_train_ca.csv", index=False)
y_train_ca.to_csv("y_train_ca.csv", index=False)
X_test_ca.to_csv("X_test_ca.csv", index=False)
y_test_ca.to_csv("y_test_ca.csv", index=False)

#### 5- Adaboost

In [185]:
X_train_ada, y_train_ada = pipeline_tree_sensitive.fit(df_train)
X_test_ada, y_test_ada = pipeline_tree_sensitive.transform(df_test)

In [186]:
X_train_ada.to_csv("X_train_ada.csv", index=False)
y_train_ada.to_csv("y_train_ada.csv", index=False)
X_test_ada.to_csv("X_test_ada.csv", index=False)
y_test_ada.to_csv("y_test_ada.csv", index=False)

# Regression

In [230]:
df2 = pd.read_csv('credit_risk_dataset.csv')
df2.shape

(32581, 12)

In [231]:
df2=df2.drop_duplicates()
df2.shape

(32416, 12)

In [232]:
df2.isna().sum()

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              887
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3095
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

In [341]:
class OutlierHandler:
    def __init__(self, columns):
        self.columns = columns
        self.bounds = {}

    def fit(self, df):
        for col in self.columns:
            if col not in df.columns:
                continue

            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1

            self.bounds[col] = {"LOF": Q1 - 3 * IQR, "UOF": Q3 + 3 * IQR}
        return self

    def transform(self, df):
        df = df.copy()
        for col in self.columns:
            if col not in df.columns or col not in self.bounds:
                continue

            LOF = self.bounds[col]["LOF"]
            UOF = self.bounds[col]["UOF"]

            df[col + "_flag"] = df[col].apply(lambda x: 1 if x < LOF or x > UOF else 0)
            df[col] = df[col].clip(lower=LOF, upper=UOF)

        return df


class FullDataPipeline:
    def __init__(
        self,
        outlier_cols,
        apply_scaling=False,
        handle_outliers=True,
        apply_encoding=True,
        drop_missing=True
    ):
        self.apply_scaling = apply_scaling
        self.handle_outliers = handle_outliers
        self.apply_encoding = apply_encoding
        self.drop_missing = drop_missing

        self.outlier_handler = OutlierHandler(outlier_cols) if handle_outliers else None

        try:
            self.ohe = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
        except TypeError:
            self.ohe = OneHotEncoder(drop="first", handle_unknown="ignore", sparse=False)

        self.numeric_scaler = None
        self.train_columns = None

        self.categorical_cols = [
            "person_home_ownership",
            "loan_intent",
            "cb_person_default_on_file",
            "loan_grade"
        ]

        self.required_cols = [
            "person_age",
            "person_income",
            "person_emp_length",                
            "loan_amnt",
            "loan_percent_income",
            "cb_person_cred_hist_length",
            "person_home_ownership",
            "loan_intent",
            "cb_person_default_on_file",
            "loan_grade"]

    def _validate_columns(self, df, stage="transform"):
        missing = set(self.required_cols) - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns at {stage}: {missing}")

    def _clean_basic(self, df):
        df = df.copy()
        df.dropna(how="all", inplace=True)
        df.drop(columns=["loan_status"], errors="ignore", inplace=True)

        if self.drop_missing:
            df.dropna(axis=0, how="any", inplace=True)

        return df

    def _scale_numeric(self, df, fit=False):
        df = df.copy()

        if not self.apply_scaling:
            return df

        cat_cols = [c for c in self.categorical_cols if c in df.columns]
        num_cols = [c for c in df.columns if c not in cat_cols]

        if fit:
            self.numeric_scaler = StandardScaler()
            df[num_cols] = self.numeric_scaler.fit_transform(df[num_cols])
        else:
            df[num_cols] = self.numeric_scaler.transform(df[num_cols])

        return df

    def _encode(self, df, fit=False):
        cat_cols = [c for c in self.categorical_cols if c in df.columns]
        num_cols = [c for c in df.columns if c not in cat_cols]

        df_num = df[num_cols].copy()

        if not self.apply_encoding:
            if len(cat_cols) == 0:
                return df_num
            return pd.concat([df_num, df[cat_cols].copy()], axis=1)

        if len(cat_cols) == 0:
            return df_num

        if fit:
            ohe_array = self.ohe.fit_transform(df[cat_cols])
        else:
            ohe_array = self.ohe.transform(df[cat_cols])

        ohe_names = self.ohe.get_feature_names_out(cat_cols)
        df_cat = pd.DataFrame(ohe_array, columns=ohe_names, index=df.index)

        return pd.concat([df_num, df_cat], axis=1)

    def fit(self, df):
        df = self._clean_basic(df)

        if "loan_int_rate" not in df.columns:
            raise ValueError("Target column 'loan_int_rate' not found in training data")
        y = df.pop("loan_int_rate")
        
     # Drop Missing values in TARGET (y)
        mask = y.notna()
        df = df.loc[mask].reset_index(drop=True)
        y  = y.loc[mask].reset_index(drop=True)

        self._validate_columns(df, stage="fit")

        if self.handle_outliers:
            self.outlier_handler.fit(df)
            df = self.outlier_handler.transform(df)

        df = self._scale_numeric(df, fit=True)
        df = self._encode(df, fit=True)

        self.train_columns = df.columns.tolist()
        return df, y

    def transform(self, df):
        if self.train_columns is None:
            raise ValueError("Pipeline has not been fitted yet. Call fit() first.")
        if self.apply_scaling and self.numeric_scaler is None:
            raise ValueError("Scaler not fitted yet. Call fit() first.")

        df = self._clean_basic(df)

        y = None
        if "loan_int_rate" in df.columns:
            y = df.pop("loan_int_rate")
            mask = y.notna()
            df = df.loc[mask].reset_index(drop=True)
            y  = y.loc[mask].reset_index(drop=True)

        self._validate_columns(df, stage="transform")

        if self.handle_outliers:
            df = self.outlier_handler.transform(df)

        df = self._scale_numeric(df, fit=False)
        df = self._encode(df, fit=False)

        df = df.reindex(columns=self.train_columns, fill_value=0)
        return df, y

    def fit_transform(self, df):
        return self.fit(df)


In [342]:

# ═══════════════════════════════════════════════════════════════
#  Regression Pipelines
# ═══════════════════════════════════════════════════════════════
if __name__ == "__main__":

    outlier_columns = ["person_income"]

    target_col = "loan_int_rate"   

    print(" 1: Pipeline for CatBoost Regressor")
    print("=" * 60)
    pipeline_cat = FullDataPipeline(
        outlier_cols=outlier_columns,
        apply_scaling=False,
        handle_outliers=False,
        apply_encoding=False,
        drop_missing=False
    )
    print("Configuration:")
    print("  - Scaling: DISABLED")
    print("  - Missing value idropped: DISABLED")
    print("  - Outlier handling: DISABLED")
    print("  - Encoding: DISABLED (CatBoost handles categoricals natively)")
    print()

    print(" 2: Pipeline for XGBoost / LightGBM Regressor")
    print("=" * 60)
    pipeline_xgb = FullDataPipeline(
        outlier_cols=outlier_columns,
        apply_scaling=False,
        handle_outliers=False,
        apply_encoding=True,
        drop_missing=False
    )
    print("Configuration:")
    print("  - Scaling: DISABLED")
    print("  - Missing value idropped: DISABLED")
    print("  - Outlier handling: DISABLED")
    print("  - Encoding: ENABLED")
    print()

    print("3: Pipeline for Linear Regression Models")
    print("=" * 60)
    pipeline_linear = FullDataPipeline(
        outlier_cols=outlier_columns,
        apply_scaling=True,
        handle_outliers=True,
        apply_encoding=True,
        drop_missing=True
    )
    print("Configuration:")
    print("  - All preprocessing ENABLED")
    print()

    print("4: Pipeline for  TREE-BASED (ROBUST)")
    print("=" * 60)
    pipeline_tree_robust = FullDataPipeline(
        outlier_cols=outlier_columns,
        apply_scaling=False,
        handle_outliers=False,
        apply_encoding=True,
        drop_missing=True
    )
    print("Configuration:")
    print("  - Scaling: DISABLED")
    print("  - Missing value dropped: ENABLED")
    print("  - Outlier handling: DISABLED")
    print("  - Encoding: ENABLED")
    print()

    print("5: Pipeline for  ADABOOST)")
    print("=" * 60)
    pipeline_tree_sensitive = FullDataPipeline(
        outlier_cols=outlier_columns,
        apply_scaling=False,    
        drop_missing=True,    
        handle_outliers=True,
        apply_encoding=True)
        
    print("Configuration:")
    print("  - Scaling: DISABLED")
    print("  - Missing value dropped: ENABLED")
    print("  - Outlier handling: ENABLED")
    print("  - Encoding: ENABLED")
    print()

    

 1: Pipeline for CatBoost Regressor
Configuration:
  - Scaling: DISABLED
  - Missing value idropped: DISABLED
  - Outlier handling: DISABLED
  - Encoding: DISABLED (CatBoost handles categoricals natively)

 2: Pipeline for XGBoost / LightGBM Regressor
Configuration:
  - Scaling: DISABLED
  - Missing value idropped: DISABLED
  - Outlier handling: DISABLED
  - Encoding: ENABLED

3: Pipeline for Linear Regression Models
Configuration:
  - All preprocessing ENABLED

4: Pipeline for  TREE-BASED (ROBUST)
Configuration:
  - Scaling: DISABLED
  - Missing value dropped: ENABLED
  - Outlier handling: DISABLED
  - Encoding: ENABLED

5: Pipeline for  ADABOOST)
Configuration:
  - Scaling: DISABLED
  - Missing value dropped: ENABLED
  - Outlier handling: ENABLED
  - Encoding: ENABLED



In [343]:
target_col= 'loan_int_rate'
#First split → Train + Temp

df2_train, df2_test = train_test_split(
    df2,
    test_size=0.20,
    random_state=42)


#### 1- Gradient Boosting 

In [344]:
pipeline_gradient_boosting = FullDataPipeline(
    outlier_cols=outlier_columns,
    apply_scaling=False,    # for Xgboost and Lightgbm
    drop_missing=False,   
    handle_outliers=False   
)

In [345]:
X_train_gb_reg, y_train_gb_reg = pipeline_gradient_boosting.fit(df2_train)
X_test_gb_reg, y_test_gb_reg = pipeline_gradient_boosting.transform(df2_test)

In [346]:
X_train_gb_reg.columns

Index(['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
       'loan_percent_income', 'cb_person_cred_hist_length',
       'person_home_ownership_OTHER', 'person_home_ownership_OWN',
       'person_home_ownership_RENT', 'loan_intent_EDUCATION',
       'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL',
       'loan_intent_PERSONAL', 'loan_intent_VENTURE',
       'cb_person_default_on_file_Y', 'loan_grade_B', 'loan_grade_C',
       'loan_grade_D', 'loan_grade_E', 'loan_grade_F', 'loan_grade_G'],
      dtype='object')

In [347]:
X_train_gb_reg.to_csv("X_train_gb_reg.csv", index=False)
y_train_gb_reg.to_csv("y_train_gb_reg.csv", index=False)
X_test_gb_reg.to_csv("X_test_gb_reg.csv", index=False)
y_test_gb_reg.to_csv("y_test_gb_reg.csv", index=False)


#### 2- Linear Models

In [348]:
X_train_s_reg, y_train_reg= pipeline_linear.fit(df2_train)
X_test_s_reg, y_test_reg = pipeline_linear.transform(df2_test)

In [349]:
X_train_s_reg.to_csv("X_train_s_reg.csv", index=False)
y_train_reg.to_csv("y_train_reg.csv", index=False)
X_test_s_reg.to_csv("X_test_s_reg.csv", index=False)
y_test_reg.to_csv("y_test_reg.csv", index=False)

In [350]:
X_train_s_reg

,person_age,person_income,person_emp_length,loan_amnt,loan_percent_income,cb_person_cred_hist_length,person_income_flag,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,...,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,cb_person_default_on_file_Y,loan_grade_B,loan_grade_C,loan_grade_D,loan_grade_E,loan_grade_F,loan_grade_G
0,-0.274240,-0.479392,-0.922330,-0.738034,-0.562732,-0.446094,-0.11726,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.826932,1.247505,0.546735,2.238997,0.469011,-0.198321,-0.11726,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,-0.431550,-0.260104,1.036423,0.528788,0.656600,-0.446094,-0.11726,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2.400034,-0.123048,-0.432642,1.320552,1.219369,1.288318,-0.11726,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,-0.588860,-0.671270,-0.432642,-0.579681,-0.187553,-0.693867,-0.11726,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22777,-0.588860,0.370351,0.791579,-0.959728,-1.125500,-0.446094,-0.11726,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22778,0.669621,-0.397159,2.750332,-0.262976,-0.093758,0.792772,-0.11726,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22779,-0.431550,0.123651,0.057046,-0.183799,-0.468937,-0.446094,-0.11726,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22780,-0.274240,2.289126,-0.922330,1.637257,-0.281347,-0.693867,-0.11726,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


#### 3-Tree Based(ROBUST)  ModelS

In [351]:
X_train_tree_reg, y_train_tree_reg= pipeline_tree_robust.fit(df2_train)
X_test_tree_reg, y_test_tree_reg= pipeline_tree_robust.transform(df2_test)

In [352]:
X_train_tree_reg.to_csv("X_train_tree_reg.csv", index=False)
y_train_tree_reg.to_csv("y_train_tree_reg.csv", index=False)
X_test_tree_reg.to_csv("X_test_tree_reg.csv", index=False)
y_test_tree_reg.to_csv("y_test_tree_reg.csv", index=False)

In [353]:
X_train_tree_reg.columns

Index(['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
       'loan_percent_income', 'cb_person_cred_hist_length',
       'person_home_ownership_OTHER', 'person_home_ownership_OWN',
       'person_home_ownership_RENT', 'loan_intent_EDUCATION',
       'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL',
       'loan_intent_PERSONAL', 'loan_intent_VENTURE',
       'cb_person_default_on_file_Y', 'loan_grade_B', 'loan_grade_C',
       'loan_grade_D', 'loan_grade_E', 'loan_grade_F', 'loan_grade_G'],
      dtype='object')

#### 4-Catboost Model

In [354]:
X_train_ca_reg, y_train_ca_reg= pipeline_cat.fit(df2_train)
X_test_ca_reg, y_test_ca_reg= pipeline_cat.transform(df2_test) 

In [355]:
X_train_ca_reg.to_csv("X_train_ca_reg.csv", index=False)
y_train_ca_reg.to_csv("y_train_ca_reg.csv", index=False)
X_test_ca_reg.to_csv("X_test_ca_reg.csv", index=False)
y_test_ca_reg.to_csv("y_test_ca_reg.csv", index=False)

In [356]:
X_train_ca_reg.columns

Index(['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
       'loan_percent_income', 'cb_person_cred_hist_length',
       'person_home_ownership', 'loan_intent', 'cb_person_default_on_file',
       'loan_grade'],
      dtype='object')

#### 5-Adaboost Model

In [357]:
X_train_ada_reg, y_train_ada_reg= pipeline_tree_sensitive.fit(df2_train)
X_test_ada_reg, y_test_ada_reg= pipeline_tree_sensitive.transform(df2_test) 

In [358]:
X_train_ada_reg.to_csv("X_train_ada_reg.csv", index=False)
y_train_ada_reg.to_csv("y_train_ada_reg.csv", index=False)
X_test_ada_reg.to_csv("X_test_ada_reg.csv", index=False)
y_test_ada_reg.to_csv("y_test_ada_reg.csv", index=False)

In [359]:
X_train_ada_reg.columns

Index(['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
       'loan_percent_income', 'cb_person_cred_hist_length',
       'person_income_flag', 'person_home_ownership_OTHER',
       'person_home_ownership_OWN', 'person_home_ownership_RENT',
       'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT',
       'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE',
       'cb_person_default_on_file_Y', 'loan_grade_B', 'loan_grade_C',
       'loan_grade_D', 'loan_grade_E', 'loan_grade_F', 'loan_grade_G'],
      dtype='object')